In [14]:
import json

# Path to the results file
results_path = "/home/hk-project-p0022573/lmu_xjh4853/workspace/pipeline_test_msc/dataset/msc_self_instruct_step_4_with_gpt_clean_qa_v2.json"
# Load the JSON data
with open(results_path, "r") as f:
    data = json.load(f)
    
for idx, item in enumerate(data):
    speaker_a = f"Speaker_{idx}_A"
    speaker_b = f"Speaker_{idx}_B"
    item["speaker_a"] = speaker_a
    item["speaker_b"] = speaker_b

# Save the modified data back to the same file
with open(results_path, "w") as f:
    json.dump(data, f, indent=2)



In [3]:

# Only save the first 100 items
split_data = data[:10]
filename = "dataset/msc_self_instruct_step_4_with_gpt_clean_qa_v2_first10.json"
with open(filename, "w") as f_out:
    json.dump(split_data, f_out, indent=2)
print(f"First 10 items saved to {filename}")

First 10 items saved to dataset/msc_self_instruct_step_4_with_gpt_clean_qa_v2_first10.json


In [10]:
from qdrant_client import QdrantClient

# Initialize a local Qdrant client with the specified path
qdrant_path = "/home/hk-project-p0022573/lmu_xjh4853/workspace/pipeline_test_msc/qdrants/add_base_llama_msc_test_2"
client = QdrantClient(path=qdrant_path)

print(f"Qdrant client initialized with local path: {qdrant_path}")


Qdrant client initialized with local path: /home/hk-project-p0022573/lmu_xjh4853/workspace/pipeline_test_msc/qdrants/add_base_llama_msc_test_2


In [11]:
# Check if the client is connected and print some info about the collections

try:
    collections = client.get_collections()
    print("Collections in Qdrant:")
    for collection in collections.collections:
        print(f"- {collection.name}")
except Exception as e:
    print(f"Error fetching collections: {e}")

# Optionally, check the status of a specific collection if you know its name
if collections.collections:
    first_collection = collections.collections[0].name
    try:
        info = client.get_collection(first_collection)
        print(f"Info for collection '{first_collection}':")
        print(info)
    except Exception as e:
        print(f"Error fetching info for collection '{first_collection}': {e}")


Collections in Qdrant:
- add_data
Info for collection 'add_data':
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=68 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=True, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=WalCon

In [13]:
# List all points (vectors) for user_id "Speaker_0_A" in all collections

from qdrant_client.models import Filter, FieldCondition, MatchValue

user_id_to_search = "Speaker_0_A"

all_points = []

try:
    collections = client.get_collections()
    for collection in collections.collections:
        collection_name = collection.name
        print(f"\nSearching collection: {collection_name}")
        try:
            # Use Qdrant's Filter object instead of a dict for scroll_filter
            scroll_result = client.scroll(
                collection_name=collection_name,
                scroll_filter=Filter(
                    must=[
                        FieldCondition(
                            key="user_id",
                            match=MatchValue(value=user_id_to_search)
                        )
                    ]
                ),
                limit=1000  # adjust as needed
            )
            points = scroll_result[0]
            if points:
                print(f"Found {len(points)} points for user_id '{user_id_to_search}' in '{collection_name}':")
                for pt in points:
                    print(f"ID: {pt.id}, Payload: {pt.payload}")
                all_points.extend(points)
            else:
                print(f"No points found for user_id '{user_id_to_search}' in '{collection_name}'.")
        except Exception as e:
            print(f"Error searching collection '{collection_name}': {e}")
except Exception as e:
    print(f"Error listing collections: {e}")

print(f"\nTotal points found for user_id '{user_id_to_search}': {len(all_points)}")



Searching collection: add_data
Found 2 points for user_id 'Speaker_0_A' in 'add_data':
ID: 282ff136-a7b6-4f61-b70e-89986dcdabe9, Payload: {'timestamp': '7 days 8 hours ago', 'user_id': 'Speaker_0_A', 'data': 'User has two dogs', 'hash': '6a2b1f587dce7e69af2e1f10fd3a34f0', 'created_at': '2025-09-27T05:37:20.463291-07:00'}
ID: 60309a62-e307-4762-ac49-c1c1f7378787, Payload: {'timestamp': '7 days 8 hours ago', 'user_id': 'Speaker_0_A', 'data': 'Works in a homeless shelter', 'hash': 'dd1047fb1faa91eb34d6015fb325e409', 'created_at': '2025-09-27T05:38:45.267047-07:00'}

Total points found for user_id 'Speaker_0_A': 2


In [ ]:
import json

json_path = "/home/hk-project-p0022573/lmu_xjh4853/workspace/pipeline_test_msc/dataset/msc_self_instruct_step_4_with_gpt_clean_qa_v2.json"

with open(json_path, "r") as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} conversations from {json_path}")

# Optionally, print a preview of the first item
if dataset:
    print("First conversation example:")
    print(json.dumps(dataset[0], indent=2))
